In [1]:
#!/usr/bin/env python
# coding: utf-8
"""
================================================================================
NQ ORB / IB — COMPLETE ANALYSIS PIPELINE
================================================================================

Every analysis from the project, in one runnable script.

  STAGE 1   load 1-min OHLCV
  STAGE 2   day-level features (sessions, ORB-15/30, IB, prior day, volume, VWAP)
  STAGE 3   09:30-09:45 microstructure features
  STAGE 4   path caches
  STAGE 5   concordance + the mechanism
  STAGE 6   IB extreme by ORB close position (deciles, joint grid, survival)
  STAGE 7   MTF-style break tables by close position
  STAGE 8   randomization null + open-position ladder
  STAGE 9   EV of the 10:00 barrier bet
  STAGE 10  pre-09:30 screen (FDR, walk-forward)
  STAGE 11  microstructure screen + orthogonality test
  STAGE 12  bull/bear asymmetry
  STAGE 13  post-IB break backtest (grid, walk-forward, years, bootstrap)
  STAGE 14  full attribute rate table
  STAGE 15  exports

Dependencies: numpy, pandas, scipy.
Runtime: ~6-10 min with the null on, ~2 min with RUN_NULL = False.

--------------------------------------------------------------------------------
TWO BUGS FIXED FROM EARLIER VERSIONS — do not reintroduce
--------------------------------------------------------------------------------
1. SESSION SHIFT.  Use (idx + 6h).normalize(), not (idx - 7h).
   With -7h the evening AFTER day D's close lands in day D's own session, so
   every overnight feature contains post-close data. Forward-looking.

2. SESSION WINDOWS.  Asia 18:00-02:00 ET, London 02:00-08:00 ET.
   An earlier version used London 03:00-09:30, which CONTAINS the 08:00 candle
   and makes "08:00 swept the London high" structurally impossible to fire.

Also note: windows are half-open, [09:30, 10:30). pandas .loc[t0:t1] is inclusive
on BOTH ends and silently pulls the 10:30 bar into the IB.
================================================================================
"""

import os
import numpy as np
import pandas as pd
from scipy import stats as sps
import warnings
warnings.filterwarnings("ignore")

# ============================================================================
# CONFIG
# ============================================================================

DATA_PATH = r"C:\Users\tammm\NQ and ES analysis\nq_15y_1min.csv"
OUT_DIR   = r"C:\Users\tammm\NQ and ES analysis\output"

START_DATE, END_DATE = "2013-01-01", "2026-12-31"
SPLIT_VAL, SPLIT_OOS = "2019-01-01", "2023-01-01"

VOL_LOOKBACK = 60          # sessions, trailing median of prior-day RTH range %
TICK, SLIP_TICKS = 0.25, 1.0
STOP_MULT, TARG_MULT = 0.30, 1.00
FDR_Q = 0.10
SEED = 11

RUN_NULL = True            # stages 8-9; the slow part
NSIM = 300

STAGES = set(range(1, 16))  # e.g. {1,2,3,4,5,6} to run a subset

RNG = np.random.default_rng(SEED)
pd.set_option("display.width", 250)
M = lambda h, m=0: h * 60 + m


def hdr(t):
    print("\n" + "=" * 104 + f"\n{t}\n" + "=" * 104, flush=True)


def sub(t):
    print("\n" + "-" * 104 + f"\n  {t}\n" + "-" * 104, flush=True)


def wilson(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    den = 1 + z * z / n
    c = (p + z * z / (2 * n)) / den
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return (100 * (c - h), 100 * (c + h))


def block_bootstrap(x, block=20, B=5000):
    """Stationary block bootstrap. Daily P&L is autocorrelated through vol
    clustering; IID resampling gives intervals 1.5-2x too narrow."""
    x = np.asarray(x, float)
    n = len(x)
    if n < 30:
        return np.nan, np.nan, np.nan
    out = np.empty(B)
    for i in range(B):
        idx = []
        while len(idx) < n:
            s0 = RNG.integers(0, n)
            idx.extend(range(s0, min(s0 + block, n)))
        out[i] = x[np.array(idx[:n])].mean()
    lo, hi = np.percentile(out, [2.5, 97.5])
    return lo, hi, (out <= 0).mean()


# --- logistic regression (Newton-Raphson) ---------------------------------
def logit_fit(X, y, iters=60):
    X = np.c_[np.ones(len(X)), X]
    w = np.zeros(X.shape[1])
    for _ in range(iters):
        p = 1 / (1 + np.exp(-np.clip(X @ w, -30, 30)))
        W = np.clip(p * (1 - p), 1e-8, None)
        try:
            step = np.linalg.solve(-(X.T * W) @ X, -(X.T @ (y - p)))
        except np.linalg.LinAlgError:
            break
        w = w + step
        if np.max(np.abs(step)) < 1e-9:
            break
    return w


def logit_pred(X, w):
    return 1 / (1 + np.exp(-np.clip(np.c_[np.ones(len(X)), X] @ w, -30, 30)))


def auc_of(y, p):
    o = np.argsort(p)
    yy = np.asarray(y)[o]
    a, b = yy.sum(), len(yy) - yy.sum()
    if a == 0 or b == 0:
        return np.nan
    r = np.arange(1, len(yy) + 1)
    return (r[yy == 1].sum() - a * (a + 1) / 2) / (a * b)


def loglik(y, p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))


def bh_fdr(pvals, q=FDR_Q):
    """Benjamini-Hochberg. Returns boolean mask of survivors."""
    p = np.asarray(pvals, float)
    order = np.argsort(p)
    m = len(p)
    thresh = (np.arange(1, m + 1) / m) * q
    passed = p[order] <= thresh
    cut = p[order][passed].max() if passed.any() else -1
    return p <= cut


# ============================================================================
# STAGE 1 — LOAD
# ============================================================================

hdr("STAGE 1 — LOADING 1-MIN OHLCV")
print(f"  {DATA_PATH}")
df = pd.read_csv(DATA_PATH, engine="c")
df["Datetime"] = pd.to_datetime(df["Date"] + " " + df["Time"], format="%Y-%m-%d %H:%M:%S")
df = df.set_index("Datetime").sort_index().loc[START_DATE:END_DATE]
print(f"  {len(df):,} bars  |  {df.index[0]} -> {df.index[-1]}")

O = df["Open"].to_numpy(np.float64)
H = df["High"].to_numpy(np.float64)
L = df["Low"].to_numpy(np.float64)
C = df["Close"].to_numpy(np.float64)
V = df["Volume"].to_numpy(np.float64)
TP = (H + L + C) / 3.0
MIN = df.index.hour.to_numpy() * 60 + df.index.minute.to_numpy()

# FIX #1: +6h, so 18:00 rolls into the NEXT trading day
SESS = (df.index + pd.Timedelta(hours=6)).normalize().to_numpy()
UNIQ, STARTS = np.unique(SESS, return_index=True)
ENDS = np.r_[STARTS[1:], len(df)]
print(f"  {len(UNIQ):,} session labels")


def win(s, e, m0, m1):
    """Half-open window [m0, m1) inside session slice [s, e)."""
    mm = MIN[s:e]
    return np.nonzero((mm >= m0) & (mm < m1))[0] + s


def wrapwin(s, e, m0, m1):
    """Window crossing midnight, e.g. 18:00 -> 02:00."""
    return np.r_[win(s, e, m0, M(24)), win(s, e, 0, m1)]


def blockstats(k, pre):
    if k.size < 10:
        return {}
    hi, lo = H[k].max(), L[k].min()
    op, cl = O[k[0]], C[k[-1]]
    ih = int(np.argmax(H[k] >= hi))
    il = int(np.argmax(L[k] <= lo))
    rng = hi - lo
    vol = V[k].sum()
    return {
        f"{pre}_h": hi, f"{pre}_l": lo, f"{pre}_o": op, f"{pre}_c": cl,
        f"{pre}_rng": rng,
        f"{pre}_rng_pct": rng / op * 100 if op else np.nan,
        f"{pre}_pos": (cl - lo) / rng if rng > 0 else np.nan,
        f"{pre}_dir": 1.0 if cl > op else 0.0,
        f"{pre}_eflow": 1.0 if il < ih else 0.0,
        f"{pre}_t_hi": float(ih), f"{pre}_t_lo": float(il),
        f"{pre}_vol": vol,
        f"{pre}_vwap": (TP[k] * V[k]).sum() / vol if vol > 0 else np.nan,
    }


def first_break(k, hi, lo):
    """First bar in k breaking hi or lo. Running extremes are monotone, so
    searchsorted finds the crossing in log time."""
    if k.size == 0:
        return None, np.nan
    rmax = np.maximum.accumulate(H[k])
    rmin = np.minimum.accumulate(L[k])
    iu = int(np.searchsorted(rmax, hi, "right")) if rmax[-1] > hi else 10 ** 9
    idn = int(np.searchsorted(-rmin, -lo, "right")) if rmin[-1] < lo else 10 ** 9
    if iu == idn == 10 ** 9:
        return "neither", np.nan
    if iu < idn:
        return "high", float(iu)
    if idn < iu:
        return "low", float(idn)
    return "same_bar", float(iu)


# ============================================================================
# STAGE 2 + 3 — DAY FEATURES AND MICROSTRUCTURE
# ============================================================================

hdr("STAGE 2/3 — BUILDING DAY-LEVEL AND MICROSTRUCTURE FEATURES")
print("  Asia 18:00-02:00 | London 02:00-08:00 | 08:00 candle | 08:00-09:30 | 06:00-09:00")

rows = []
for j in range(len(UNIQ)):
    s, e = STARTS[j], ENDS[j]
    if e - s < 400:
        continue
    k_rth = win(s, e, M(9, 30), M(16))
    if k_rth.size < 300:
        continue

    r = {"session": UNIQ[j],
         "rth_open": O[k_rth[0]], "rth_close": C[k_rth[-1]],
         "rth_high": H[k_rth].max(), "rth_low": L[k_rth].min()}
    ih = int(np.argmax(H[k_rth] >= r["rth_high"]))
    il = int(np.argmax(L[k_rth] <= r["rth_low"]))
    r["rth_hod_first"] = 1.0 if ih < il else 0.0

    # FIX #2: correct session windows
    r.update(blockstats(wrapwin(s, e, M(18), M(2)), "asia"))
    r.update(blockstats(win(s, e, M(2), M(8)), "lon"))
    r.update(blockstats(np.r_[win(s, e, M(18), M(24)), win(s, e, 0, M(9, 30))], "on"))
    r.update(blockstats(win(s, e, M(6), M(9)), "pm69"))
    r.update(blockstats(win(s, e, M(8), M(9)), "h8"))
    r.update(blockstats(win(s, e, M(8), M(9, 30)), "nypm"))
    r.update(blockstats(win(s, e, M(9, 30), M(9, 45)), "o15"))
    r.update(blockstats(win(s, e, M(9, 30), M(10)), "o30"))
    r.update(blockstats(win(s, e, M(9, 30), M(10, 30)), "ib"))

    need = ("asia_h", "lon_h", "on_h", "pm69_h", "h8_h", "nypm_h", "o15_h", "o30_h", "ib_h")
    if any(x not in r for x in need):
        continue

    # 09:30 open position within each prior structure
    o = r["rth_open"]
    for pre in ("asia", "lon", "on", "pm69", "h8", "nypm"):
        rr = r.get(f"{pre}_rng", np.nan)
        if rr and rr > 0:
            r[f"open_pos_{pre}"] = (o - r[f"{pre}_l"]) / rr
            r[f"open_vs_mid_{pre}"] = (o - (r[f"{pre}_h"] + r[f"{pre}_l"]) / 2) / rr
        vw = r.get(f"{pre}_vwap", np.nan)
        if np.isfinite(vw) and vw > 0:
            r[f"open_vs_vwap_{pre}"] = (o - vw) / o * 100

    # sweeps
    r["lon_swept_asia_hi"] = float(r["lon_h"] > r["asia_h"])
    r["lon_swept_asia_lo"] = float(r["lon_l"] < r["asia_l"])
    r["h8_swept_lon_hi"] = float(r["h8_h"] > r["lon_h"])
    r["h8_swept_lon_lo"] = float(r["h8_l"] < r["lon_l"])
    r["nypm_swept_lon_hi"] = float(r["nypm_h"] > r["lon_h"])
    r["nypm_swept_lon_lo"] = float(r["nypm_l"] < r["lon_l"])
    r["nypm_swept_asia_hi"] = float(r["nypm_h"] > r["asia_h"])
    r["nypm_swept_asia_lo"] = float(r["nypm_l"] < r["asia_l"])
    r["o15_swept_on_hi"] = float(r["o15_h"] > r["on_h"])
    r["o15_swept_on_lo"] = float(r["o15_l"] < r["on_l"])

    # nested: ORB-30 close inside the ORB-15 range
    if r["o15_rng"] > 0:
        r["o30c_in_o15"] = (r["o30_c"] - r["o15_l"]) / r["o15_rng"]

    # do the ORB extremes survive to become the IB extremes?
    r["o15_hi_survives"] = float(r["ib_h"] <= r["o15_h"])
    r["o15_lo_survives"] = float(r["ib_l"] >= r["o15_l"])
    r["o30_hi_survives"] = float(r["ib_h"] <= r["o30_h"])
    r["o30_lo_survives"] = float(r["ib_l"] >= r["o30_l"])
    r["o15_both_survive"] = r["o15_hi_survives"] * r["o15_lo_survives"]
    r["o30_both_survive"] = r["o30_hi_survives"] * r["o30_lo_survives"]

    # live-safe break windows
    r["o15_brk_by10"], r["o15_brk_by10_t"] = first_break(win(s, e, M(9, 45), M(10)), r["o15_h"], r["o15_l"])
    r["o15_brk_1030"], _ = first_break(win(s, e, M(9, 45), M(10, 30)), r["o15_h"], r["o15_l"])
    r["o15_brk_1600"], _ = first_break(win(s, e, M(9, 45), M(16)), r["o15_h"], r["o15_l"])
    r["o30_brk"], r["o30_brk_t"] = first_break(win(s, e, M(10), M(10, 30)), r["o30_h"], r["o30_l"])
    r["o30_brk_1100"], _ = first_break(win(s, e, M(10), M(11)), r["o30_h"], r["o30_l"])
    r["o30_brk_1600"], _ = first_break(win(s, e, M(10), M(16)), r["o30_h"], r["o30_l"])

    # post-IB
    k_post = win(s, e, M(10, 30), M(16))
    if k_post.size < 200:
        continue
    side, tmin = first_break(k_post, r["ib_h"], r["ib_l"])
    r["ib_brk"], r["ib_brk_t"] = side, tmin
    r["brk_high"] = 1.0 if side == "high" else (0.0 if side == "low" else np.nan)
    k1h = win(s, e, M(10, 30), M(11, 30))
    r["ib_h_1h"] = bool((H[k1h] > r["ib_h"]).any())
    r["ib_l_1h"] = bool((L[k1h] < r["ib_l"]).any())
    r["ib_h_ses"] = bool((H[k_post] > r["ib_h"]).any())
    r["ib_l_ses"] = bool((L[k_post] < r["ib_l"]).any())
    if side in ("high", "low"):
        kk = k_post[int(tmin):]
        if kk.size > 5:
            if side == "high":
                r["brk_mfe"] = (H[kk].max() - r["ib_h"]) / r["ib_h"] * 100
                r["brk_mae"] = max(0.0, r["ib_h"] - L[kk].min()) / r["ib_h"] * 100
            else:
                r["brk_mfe"] = (r["ib_l"] - L[kk].min()) / r["ib_l"] * 100
                r["brk_mae"] = max(0.0, H[kk].max() - r["ib_l"]) / r["ib_l"] * 100

    # ---- STAGE 3: 09:30-09:45 microstructure ----
    k15 = win(s, e, M(9, 30), M(9, 45))
    if k15.size >= 15:
        oo, hh, ll, cc, vv = O[k15], H[k15], L[k15], C[k15], V[k15]
        op, cl = oo[0], cc[-1]
        hi, lo = hh.max(), ll.min()
        rng = hi - lo
        if rng > 0:
            for kk_, (i0, i1) in enumerate([(0, 5), (5, 10), (10, 15)], 1):
                sh, sl = hh[i0:i1].max(), ll[i0:i1].min()
                r[f"b{kk_}_pos"] = (cc[i1 - 1] - sl) / (sh - sl) if sh > sl else np.nan
                r[f"b{kk_}_ret"] = (cc[i1 - 1] - oo[i0]) / oo[i0] * 100
                r[f"b{kk_}_rng"] = (sh - sl) / oo[i0] * 100
            r["m1_pos"] = (cc[0] - ll[0]) / (hh[0] - ll[0]) if hh[0] > ll[0] else np.nan
            r["m1_ret"] = (cc[0] - oo[0]) / oo[0] * 100
            r["m3_pos"] = ((cc[2] - ll[:3].min()) / (hh[:3].max() - ll[:3].min())
                           if hh[:3].max() > ll[:3].min() else np.nan)
            r["m5_ret"] = (cc[4] - op) / op * 100
            r["o15_ret"] = (cl - op) / op * 100
            r["open_pos_in_rng"] = (op - lo) / rng
            r["mfe_from_open"] = (hi - op) / op * 100
            r["mae_from_open"] = (op - lo) / op * 100
            r["mfe_mae_ratio"] = (hi - op) / (op - lo) if op > lo else np.nan
            ih2 = int(np.argmax(hh >= hi)); il2 = int(np.argmax(ll <= lo))
            r["t_hi_15"], r["t_lo_15"] = float(ih2), float(il2)
            r["extreme_spread"] = (ih2 - il2) / 15.0
            r["hi_in_last5"] = float(ih2 >= 10); r["lo_in_last5"] = float(il2 >= 10)
            r["hi_in_first5"] = float(ih2 < 5); r["lo_in_first5"] = float(il2 < 5)
            steps = np.abs(np.diff(np.r_[op, cc])); tot = steps.sum()
            r["efficiency"] = abs(cl - op) / tot if tot > 0 else np.nan
            r["range_eff"] = abs(cl - op) / rng
            r["n_up_bars"] = float((cc > oo).sum())
            r["bar_rng_max"] = (hh - ll).max() / op * 100
            r["bar_rng_std"] = ((hh - ll) / op * 100).std()
            dsig = np.sign(cc - oo); run = mx = 1
            for k2 in range(1, len(dsig)):
                run = run + 1 if dsig[k2] == dsig[k2 - 1] and dsig[k2] != 0 else 1
                mx = max(mx, run)
            r["max_run"] = float(mx)
            c2c = np.std(np.diff(np.r_[op, cc]) / op * 100)
            pk = np.mean(((hh - ll) / op * 100) ** 2) ** .5
            r["chop"] = pk / c2c if c2c > 0 else np.nan
            tv = vv.sum()
            if tv > 0:
                r["vol_front"] = vv[:5].sum() / tv
                r["vol_back"] = vv[10:].sum() / tv
                r["vol_m1_share"] = vv[0] / tv
                r["vol_up_share"] = vv[cc > oo].sum() / tv
                vwap = ((hh + ll + cc) / 3 * vv).sum() / tv
                r["close_vs_vwap"] = (cl - vwap) / op * 100
                r["vwap_pos"] = (vwap - lo) / rng
    rows.append(r)

d = pd.DataFrame(rows).set_index("session").sort_index()

# ---- derived: prior day, trailing vol, ratios ----
d["prev_h"] = d["rth_high"].shift(1); d["prev_l"] = d["rth_low"].shift(1)
d["prev_c"] = d["rth_close"].shift(1); d["prev_o"] = d["rth_open"].shift(1)
d["prev_rng"] = d["prev_h"] - d["prev_l"]
d["prev_rng_pct"] = d["prev_rng"] / d["prev_c"] * 100
d["prev_dir"] = (d["prev_c"] > d["prev_o"]).astype(float)
d["prev_close_pos"] = (d["prev_c"] - d["prev_l"]) / d["prev_rng"]
d["vol60"] = d["prev_rng_pct"].rolling(VOL_LOOKBACK, min_periods=VOL_LOOKBACK // 2).median()
d["open_pos_prev"] = ((d["rth_open"] - d["prev_l"]) / d["prev_rng"]).clip(-1.5, 2.5)
d["gap_rel"] = (d["rth_open"] / d["prev_c"] - 1) * 100 / d["vol60"]
d["dist_prev_h"] = (d["prev_h"] / d["rth_open"] - 1) * 100 / d["vol60"]
d["dist_prev_l"] = (d["rth_open"] / d["prev_l"] - 1) * 100 / d["vol60"]
d["on_swept_prev_hi"] = (d["on_h"] > d["prev_h"]).astype(float)
d["on_swept_prev_lo"] = (d["on_l"] < d["prev_l"]).astype(float)
d["prev_ib_eflow"] = d["ib_eflow"].shift(1)
d["prev_brk_high"] = d["brk_high"].shift(1)
for pre in ("asia", "lon", "on", "pm69", "h8", "nypm", "o15", "o30", "ib"):
    d[f"{pre}_rng_rel"] = d[f"{pre}_rng_pct"] / d["vol60"]
    med = d[f"{pre}_vol"].rolling(60, min_periods=30).median().shift(1)
    d[f"{pre}_vol_rel"] = d[f"{pre}_vol"] / med
d["lon_vs_asia_rng"] = d["lon_rng"] / d["asia_rng"]
d["nypm_vs_lon_rng"] = d["nypm_rng"] / d["lon_rng"]
d["h8_vs_lon_rng"] = d["h8_rng"] / d["lon_rng"]
d["h8_vs_pm69_rng"] = d["h8_rng"] / d["pm69_rng"]
d["on_vol_skew"] = d["lon_vol"] / (d["asia_vol"] + d["lon_vol"])
d["b2_vs_b1_rng"] = d["b2_rng"] / d["b1_rng"]
d["b3_vs_b1_rng"] = d["b3_rng"] / d["b1_rng"]
d["o15_ret_rel"] = d["o15_ret"] / d["vol60"]
d["m1_ret_rel"] = d["m1_ret"] / d["vol60"]
d["m5_ret_rel"] = d["m5_ret"] / d["vol60"]
wk = pd.Series(d.index).dt.isocalendar()
d["_wk"] = (wk["year"].astype(int) * 100 + wk["week"].astype(int)).to_numpy()
d["open_vs_wvwap"] = ((d["rth_open"] - d.groupby("_wk")["on_vwap"]
                       .transform(lambda s: s.expanding().mean()).shift(1))
                      / d["rth_open"] * 100)

# targets
d["T1_ny_bull"] = (d["rth_close"] > d["rth_open"]).astype(float)
d["T2_ib_eflow"] = d["ib_eflow"]
d["T3_brk_high"] = d["brk_high"]
d["T4_hod_first"] = d["rth_hod_first"]
d["period"] = np.where(d.index < np.datetime64(SPLIT_VAL), "IS",
               np.where(d.index < np.datetime64(SPLIT_OOS), "VAL", "OOS"))
d["szq"] = pd.qcut(d["ib_rng_rel"], 4, labels=["Q1", "Q2", "Q3", "Q4"])

print(f"  {len(d):,} days x {d.shape[1]} cols  |  "
      + "  ".join(f"{k}={v}" for k, v in d["period"].value_counts().items()))
print("  base rates: " + "  ".join(
    f"{t}={d[t].mean()*100:.1f}%" for t in ["T1_ny_bull", "T2_ib_eflow", "T3_brk_high", "T4_hod_first"]))


# ============================================================================
# STAGE 4 — PATH CACHES
# ============================================================================

hdr("STAGE 4 — CACHING 1-MIN PATHS")

def cache_running(m0, m1, min_cov=0.8):
    """{session: (running_max_ret, running_min_ret, close)} relative to the
    window's opening price. Monotone -> searchsortable."""
    msk = (MIN >= m0) & (MIN < m1)
    ss = SESS[msk]
    hh, ll, cc = H[msk], L[msk], C[msk]
    u, st = np.unique(ss, return_index=True)
    en = np.r_[st[1:], len(hh)]
    out = {}
    for i, k in enumerate(u):
        a, b = st[i], en[i]
        if b - a < (m1 - m0) * min_cov:
            continue
        c0 = cc[a]
        out[k] = (np.maximum.accumulate(hh[a:b]) / c0 - 1.0,
                  np.minimum.accumulate(ll[a:b]) / c0 - 1.0, cc[b - 1])
    return out


def cache_raw(m0, m1, min_cov=0.8):
    msk = (MIN >= m0) & (MIN < m1)
    ss = SESS[msk]
    hh, ll, cc = H[msk], L[msk], C[msk]
    u, st = np.unique(ss, return_index=True)
    en = np.r_[st[1:], len(hh)]
    return {k: (hh[st[i]:en[i]], ll[st[i]:en[i]], cc[st[i]:en[i]])
            for i, k in enumerate(u) if en[i] - st[i] >= (m1 - m0) * min_cov}


PATH_POST = cache_raw(M(10, 30), M(16))
_dropped = len(d) - len(set(d.index) & set(PATH_POST))
print(f"  10:30-16:00 raw paths: {len(PATH_POST):,}   |  {_dropped:,} of the {len(d):,} feature days")
print("  have no qualifying path and are excluded from the backtest: half-sessions (Jul 3,")
print("  Dec 24, day after Thanksgiving) close at 13:15, so a 16:00 time-stop is meaningless.")


# ============================================================================
# STAGE 5 — CONCORDANCE AND THE MECHANISM
# ============================================================================

if 5 in STAGES:
    hdr("STAGE 5 — DOES THE ORB EXTREME ORDER SURVIVE INTO THE IB?")
    dd = d[d["ib_eflow"].notna()]
    sub("5a. carry-forward concordance")
    for tag in ("o15", "o30"):
        ct = pd.crosstab(dd[f"{tag}_eflow"].map({1.0: "low", 0.0: "high"}),
                         dd["ib_eflow"].map({1.0: "low", 0.0: "high"}))
        print(f"\n  {tag}_eflow -> ib_eflow  (n={ct.values.sum():,})")
        print("  " + ct.to_string().replace("\n", "\n  "))
        print(f"  agreement: {np.trace(ct.values)/ct.values.sum()*100:.1f}%")

    sub("5b. the ORB-30 break IS the determination")
    for brk in ("high", "low", "neither"):
        g = dd[dd["o30_brk"] == brk]
        if len(g) < 20:
            continue
        print(f"  o30_brk={brk:<8} n={len(g):5d}   ib_ef=high {(1-g['ib_eflow'].mean())*100:5.1f}%"
              f"   ib_ef=low {g['ib_eflow'].mean()*100:5.1f}%"
              f"   ib_ef==o30_ef {(g['ib_eflow']==g['o30_eflow']).mean()*100:6.1f}%")

    sub("5c. break frequencies")
    for c, lab in (("o15_brk_by10", "ORB-15 by 10:00"), ("o15_brk_1030", "ORB-15 by 10:30"),
                   ("o30_brk", "ORB-30 by 10:30")):
        vc = dd[c].value_counts(normalize=True) * 100
        print(f"  {lab:<20}" + "   ".join(f"{k}={v:.1f}%" for k, v in vc.items()))


# ============================================================================
# STAGE 6 — IB EXTREME BY ORB CLOSE POSITION
# ============================================================================

if 6 in STAGES:
    hdr("STAGE 6 — IB EXTREME BY ORB CLOSE POSITION")

    def decile_table(col, title):
        q = d.dropna(subset=[col]).copy()
        q["b"] = pd.qcut(q[col], 10, labels=False)
        sub(title)
        print(f"  {'dec':<5}{'range':<14}{'n':>5}{'IB low 1st':>12}{'IB high 1st':>13}"
              f"{'post-IB brk hi':>16}{'NY bull':>10}")
        for b, g in q.groupby("b"):
            rg = f"{g[col].min():.2f}-{g[col].max():.2f}"
            gb = g.dropna(subset=["brk_high"])
            print(f"  {int(b)+1:<5}{rg:<14}{len(g):5d}{g['ib_eflow'].mean()*100:11.1f}%"
                  f"{(1-g['ib_eflow'].mean())*100:12.1f}%{gb['brk_high'].mean()*100:15.1f}%"
                  f"{g['T1_ny_bull'].mean()*100:9.1f}%")
        gb = q.dropna(subset=["brk_high"])
        print(f"  {'ALL':<5}{'':<14}{len(q):5d}{q['ib_eflow'].mean()*100:11.1f}%"
              f"{(1-q['ib_eflow'].mean())*100:12.1f}%{gb['brk_high'].mean()*100:15.1f}%"
              f"{q['T1_ny_bull'].mean()*100:9.1f}%")

    decile_table("o15_pos", "6a. ORB-15 close position (known 09:45)")
    decile_table("o30_pos", "6b. ORB-30 close position (known 10:00)")

    sub("6c. joint grid — P(IB low first) by ORB-15 x ORB-30 quintile")
    q = d.dropna(subset=["o15_pos", "o30_pos"]).copy()
    q["q15"] = pd.qcut(q["o15_pos"], 5, labels=["15:Q1", "Q2", "Q3", "Q4", "Q5"])
    q["q30"] = pd.qcut(q["o30_pos"], 5, labels=["30:Q1", "Q2", "Q3", "Q4", "Q5"])
    print("\n  P(IB low first) %")
    print("  " + (pd.crosstab(q["q15"], q["q30"], values=q["ib_eflow"], aggfunc="mean") * 100
                  ).round(1).to_string().replace("\n", "\n  "))
    print("\n  n")
    print("  " + pd.crosstab(q["q15"], q["q30"]).to_string().replace("\n", "\n  "))
    print("\n  Down a column: barely moves. Across a row: swings ~85pp.")
    print("  The ORB-30 fixes the answer; the ORB-15 only helps in the middle column.")

    sub("6d. does the ORB extreme SURVIVE to become the IB extreme?")
    for col, tag, lab in (("o15_pos", "o15", "ORB-15 at 09:45"), ("o30_pos", "o30", "ORB-30 at 10:00")):
        q = d.dropna(subset=[col]).copy()
        q["b"] = pd.cut(q[col], np.arange(0, 1.0001, .2), include_lowest=True)
        print(f"\n  {lab}")
        print(f"    {'close pos':<14}{'n':>6}{'HIGH survives':>15}{'LOW survives':>14}{'BOTH':>10}")
        for b, g in q.groupby("b", observed=True):
            print(f"    {str(b):<14}{len(g):6d}{g[f'{tag}_hi_survives'].mean()*100:14.1f}%"
                  f"{g[f'{tag}_lo_survives'].mean()*100:13.1f}%{g[f'{tag}_both_survive'].mean()*100:9.1f}%")
        print(f"    {'ALL':<14}{len(q):6d}{q[f'{tag}_hi_survives'].mean()*100:14.1f}%"
              f"{q[f'{tag}_lo_survives'].mean()*100:13.1f}%{q[f'{tag}_both_survive'].mean()*100:9.1f}%")

    sub("6e. concordance by close position — U-shaped")
    for col, tag, lab in (("o15_pos", "o15", "ORB-15"), ("o30_pos", "o30", "ORB-30")):
        q = d.dropna(subset=[col]).copy()
        q["b"] = pd.cut(q[col], np.arange(0, 1.0001, .2), include_lowest=True)
        print(f"\n  {lab} — P({tag}_eflow == ib_eflow)")
        for b, g in q.groupby("b", observed=True):
            print(f"    {str(b):<14}{len(g):6d}{(g[f'{tag}_eflow']==g['ib_eflow']).mean()*100:10.1f}%")
        print(f"    {'ALL':<14}{len(q):6d}{(q[f'{tag}_eflow']==q['ib_eflow']).mean()*100:10.1f}%")


# ============================================================================
# STAGE 7 — MTF-STYLE BREAK TABLES
# ============================================================================

if 7 in STAGES:
    hdr("STAGE 7 — MTF-STYLE: CLOSE POSITION -> WHICH SIDE OF THE RANGE BREAKS FIRST")

    def mtf_table(posc, outc, title):
        q = d[[posc, outc]].dropna()
        q = q[q[outc].isin(["high", "low", "neither"])]
        b = pd.cut(q[posc], np.arange(0, 1.0001, .1), include_lowest=True)
        sub(title)
        print(f"  {'close pos':<15}{'n':>6}{'brk HIGH':>11}{'brk LOW':>10}{'neither':>10}{'up|resolved':>14}")
        for k, g in q.groupby(b, observed=True):
            h_ = (g[outc] == "high").mean() * 100
            l_ = (g[outc] == "low").mean() * 100
            res = h_ / (h_ + l_) * 100 if h_ + l_ > 0 else np.nan
            print(f"  {str(k):<15}{len(g):6d}{h_:10.1f}%{l_:9.1f}%"
                  f"{(g[outc]=='neither').mean()*100:9.1f}%{res:13.1f}%")
        h_ = (q[outc] == "high").mean() * 100
        l_ = (q[outc] == "low").mean() * 100
        print(f"  {'ALL':<15}{len(q):6d}{h_:10.1f}%{l_:9.1f}%"
              f"{(q[outc]=='neither').mean()*100:9.1f}%{h_/(h_+l_)*100:13.1f}%")

    mtf_table("o15_pos", "o15_brk_by10", "7a. ORB-15 -> breaks 09:45-10:00")
    mtf_table("o15_pos", "o15_brk_1030", "7b. ORB-15 -> breaks 09:45-10:30")
    mtf_table("o15_pos", "o15_brk_1600", "7c. ORB-15 -> breaks 09:45-16:00")
    mtf_table("o30_pos", "o30_brk", "7d. ORB-30 -> breaks 10:00-10:30")
    mtf_table("o30_pos", "o30_brk_1100", "7e. ORB-30 -> breaks 10:00-11:00")
    mtf_table("o30_pos", "o30_brk_1600", "7f. ORB-30 -> breaks 10:00-16:00")
    mtf_table("o30c_in_o15", "o15_brk_1600", "7g. NESTED: ORB-30 close inside ORB-15 -> ORB-15 breaks to 16:00")
    print("\n  Note the 'neither' column: ~25% mid-range vs ~6% at the extremes over 30 min,")
    print("  collapsing to <1% by 16:00. Given time, every ORB range breaks. The question")
    print("  was never whether, only which side first - and that is mostly geometry.")


# ============================================================================
# STAGE 8/9 — RANDOMIZATION NULL, LADDER, AND BARRIER-BET EV
# ============================================================================

if RUN_NULL and (8 in STAGES or 9 in STAGES):
    hdr("STAGE 8 — RANDOMIZATION NULL: OPEN-POSITION LADDER")
    print("  Gambler's ruin (P(up first) = open position) assumes an UNBOUNDED horizon")
    print("  and is the wrong null over a fixed window: the near barrier is nearly certain")
    print("  to be tagged while the far one often is not. Instead, replay OTHER days' real")
    print("  paths (vol-matched) against this day's real barriers.")

    LADDER = [("ORB-15 -> 10:30", M(9, 30), M(9, 45), M(9, 45), M(10, 30)),
              ("ORB-30 -> 10:30", M(9, 30), M(10), M(10), M(10, 30)),
              ("IB -> 11:30", M(9, 30), M(10, 30), M(10, 30), M(11, 30)),
              ("IB -> 16:00", M(9, 30), M(10, 30), M(10, 30), M(16))]

    summary = []
    for nm, r0, r1, w0, w1 in LADDER:
        paths = cache_running(w0, w1)
        recs = []
        for j in range(len(UNIQ)):
            s, e = STARTS[j], ENDS[j]
            if e - s < 400 or UNIQ[j] not in paths:
                continue
            kr = win(s, e, r0, r1)
            kw = win(s, e, w0, w1)
            if kr.size < 10 or kw.size < 5:
                continue
            hi, lo = H[kr].max(), L[kr].min()
            op = O[kw[0]]
            if hi <= lo:
                continue
            rmax, rmin, cl = paths[UNIQ[j]]
            upn, dnn = hi / op - 1, lo / op - 1
            iu = int(np.searchsorted(rmax, upn, "right")) if rmax[-1] > upn else 10 ** 9
            idn = int(np.searchsorted(-rmin, -dnn, "right")) if rmin[-1] < dnn else 10 ** 9
            side = ("neither" if iu == idn == 10 ** 9 else
                    "high" if iu < idn else "low" if idn < iu else "same")
            recs.append({"session": UNIQ[j], "open_pos": (op - lo) / (hi - lo),
                         "up": upn, "dn": dnn, "side": side, "rng": hi - lo,
                         "op": op, "close": cl})
        z = pd.DataFrame(recs).set_index("session").join(d[["vol60", "period"]], how="inner").dropna(subset=["vol60"])
        z = z.loc[[k for k in z.index if k in paths]]
        RM = [paths[k][0] for k in z.index]
        RN = [paths[k][1] for k in z.index]
        upv, dnv = z["up"].to_numpy(), z["dn"].to_numpy()
        vt = pd.qcut(z["vol60"], 3, labels=False).to_numpy()
        donors = {t: np.nonzero(vt == t)[0] for t in range(3)}
        n = len(z)
        su = np.zeros(n); sd = np.zeros(n); sn = np.zeros(n)
        for i in range(n):
            for k in RNG.choice(donors[vt[i]], NSIM):
                rm, rn_ = RM[k], RN[k]
                a = np.searchsorted(rm, upv[i], "right") if rm[-1] > upv[i] else 10 ** 9
                b = np.searchsorted(-rn_, -dnv[i], "right") if rn_[-1] < dnv[i] else 10 ** 9
                if a == b == 10 ** 9: sn[i] += 1
                elif a < b: su[i] += 1
                elif b < a: sd[i] += 1
                else: su[i] += .5; sd[i] += .5
        su /= NSIM; sd /= NSIM; sn /= NSIM
        z["null_up"] = su / np.where(su + sd > 0, su + sd, np.nan)
        z["null_nei"] = sn

        r = z[z["side"].isin(["high", "low"])].copy()
        r["b"] = pd.cut(r["open_pos"], [-9, .1, .3, .5, .7, .9, 9],
                        labels=["<0.1", "0.1-0.3", "0.3-0.5", "0.5-0.7", "0.7-0.9", ">0.9"])
        sub(f"{nm}   n={len(z):,}   neither: obs {(z['side']=='neither').mean()*100:.1f}%"
            f"  null {z['null_nei'].mean()*100:.1f}%")
        print(f"  {'open pos':<11}{'n':>6}{'null%':>8}{'obs%':>8}{'dev':>8}{'z':>7}{'EV(R)':>9}{'EV/risk':>9}{'t':>7}")
        for b, g in r.groupby("b", observed=True):
            if len(g) < 40:
                continue
            exp = g["null_up"].mean() * 100
            obs = (g["side"] == "high").mean() * 100
            se = np.sqrt(exp / 100 * (1 - exp / 100) / len(g)) * 100
            p = g["open_pos"].clip(0, 1)
            long_ = g["open_pos"].mean() >= .5
            near = "high" if long_ else "low"
            rew = (1 - p) if long_ else p
            rsk = p if long_ else (1 - p)
            flat = ((g["close"] - g["op"]) / g["rng"]) * (1 if long_ else -1)
            pnl = np.where(g["side"] == near, rew, np.where(g["side"] != "neither", -rsk, flat))
            tt = pnl.mean() / (pnl.std(ddof=1) / np.sqrt(len(g))) if pnl.std() > 0 else np.nan
            print(f"  {str(b):<11}{len(g):6d}{exp:8.1f}{obs:8.1f}{obs-exp:+8.1f}{(obs-exp)/se:7.1f}"
                  f"{pnl.mean():+9.3f}{pnl.mean()/rsk.mean():+9.3f}{tt:7.1f}")
        eff = ((r["side"] == "high").astype(float) - r["null_up"]) * np.sign(r["open_pos"] - .5)
        eff = eff.dropna()
        t = eff.mean() / (eff.std(ddof=1) / np.sqrt(len(eff)))
        print(f"  pooled {eff.mean()*100:+.2f}pp  t={t:+.1f}  "
              f"({'continuation' if eff.mean()>0 else 'MEAN REVERSION'})")
        summary.append({"ladder": nm, "n": len(z), "pooled_pp": eff.mean() * 100, "t": t,
                        "neither_obs": (z["side"] == "neither").mean() * 100,
                        "neither_null": z["null_nei"].mean() * 100})

    sub("STAGE 9 — SUMMARY: deviation from the null, signed toward the NEAR barrier")
    LADDER_SUM = pd.DataFrame(summary)
    print("  " + LADDER_SUM.to_string(index=False, float_format=lambda v: f"{v:.2f}").replace("\n", "\n  "))
    print("\n  Every ladder resolves MORE than replayed paths. Real intraday paths are more")
    print("  directional than vol-matched random ones. That is a VOLATILITY statement, not a")
    print("  directional one - and it is why the IB break works and fading the IB does not.")
    print("  Beware the EV t-stats at extreme open positions: you win small very often and")
    print("  the loss tail is short, so the standard error badly understates the true risk.")
else:
    LADDER_SUM = None


# ============================================================================
# STAGE 10 — PRE-09:30 SCREEN
# ============================================================================

TARGETS = {"T3_brk_high": "post-IB break high", "T2_ib_eflow": "IB low 1st",
           "T1_ny_bull": "NY bull", "T4_hod_first": "HOD 1st"}

PRE_FEATS = [
 "asia_pos", "asia_dir", "asia_eflow", "asia_rng_rel", "asia_vol_rel",
 "lon_pos", "lon_dir", "lon_eflow", "lon_rng_rel", "lon_vol_rel", "lon_vs_asia_rng",
 "lon_swept_asia_hi", "lon_swept_asia_lo",
 "on_pos", "on_eflow", "on_rng_rel", "on_vol_rel", "on_vol_skew",
 "open_pos_asia", "open_pos_lon", "open_pos_on", "open_pos_pm69", "open_pos_h8", "open_pos_nypm",
 "open_vs_mid_lon", "open_vs_mid_asia", "open_vs_mid_on", "open_vs_mid_pm69",
 "open_vs_vwap_lon", "open_vs_vwap_asia", "open_vs_vwap_on", "open_vs_vwap_pm69",
 "open_vs_vwap_h8", "open_vs_vwap_nypm", "open_vs_wvwap",
 "pm69_pos", "pm69_dir", "pm69_eflow", "pm69_rng_rel", "pm69_vol_rel",
 "h8_pos", "h8_dir", "h8_eflow", "h8_rng_rel", "h8_vol_rel", "h8_vs_lon_rng", "h8_vs_pm69_rng",
 "h8_swept_lon_hi", "h8_swept_lon_lo",
 "nypm_pos", "nypm_dir", "nypm_eflow", "nypm_rng_rel", "nypm_vol_rel", "nypm_vs_lon_rng",
 "nypm_swept_lon_hi", "nypm_swept_lon_lo", "nypm_swept_asia_hi", "nypm_swept_asia_lo",
 "prev_dir", "prev_close_pos", "prev_rng_pct", "open_pos_prev", "gap_rel",
 "dist_prev_h", "dist_prev_l", "on_swept_prev_hi", "on_swept_prev_lo",
 "prev_ib_eflow", "prev_brk_high",
]
MICRO_FEATS = [
 "o15_pos", "o15_ret_rel", "o15_rng_rel", "o15_eflow", "o15_dir",
 "b1_pos", "b2_pos", "b3_pos", "b1_ret", "b2_ret", "b3_ret",
 "m1_pos", "m1_ret_rel", "m3_pos", "m5_ret_rel",
 "open_pos_in_rng", "mfe_from_open", "mae_from_open", "mfe_mae_ratio",
 "t_hi_15", "t_lo_15", "extreme_spread", "hi_in_last5", "lo_in_last5",
 "hi_in_first5", "lo_in_first5", "efficiency", "range_eff", "n_up_bars",
 "max_run", "chop", "bar_rng_max", "bar_rng_std", "b2_vs_b1_rng", "b3_vs_b1_rng",
 "vol_front", "vol_back", "vol_m1_share", "o15_vol_rel", "vol_up_share",
 "close_vs_vwap", "vwap_pos", "o15_swept_on_hi", "o15_swept_on_lo",
]

def screen(feats, label):
    dev = d[d["period"].isin(["IS", "VAL"])]
    oos = d[d["period"] == "OOS"]
    rows = []
    for t in TARGETS:
        for f in feats:
            if f not in d.columns:
                continue
            s = dev[[f, t]].dropna()
            if len(s) < 500 or s[f].std() == 0:
                continue
            x = s[f].to_numpy().reshape(-1, 1)
            y = s[t].to_numpy()
            lo, hi = np.percentile(x, [.5, 99.5])
            if hi <= lo:
                continue
            x = np.clip(x, lo, hi)
            w = logit_fit(x, y)
            lr = 2 * (loglik(y, logit_pred(x, w)) - loglik(y, np.full(len(y), y.mean())))
            so = oos[[f, t]].dropna()
            ao = (auc_of(so[t].to_numpy(), logit_pred(np.clip(so[f].to_numpy().reshape(-1, 1), lo, hi), w))
                  if len(so) > 150 else np.nan)
            rows.append({"target": t, "feature": f, "auc": auc_of(y, logit_pred(x, w)),
                         "auc_oos": ao, "p": 1 - sps.chi2.cdf(max(lr, 0), 1)})
    R = pd.DataFrame(rows)
    R["survives"] = bh_fdr(R["p"].to_numpy())
    hdr(f"{label} — {len(set(R['feature']))} features x {len(TARGETS)} targets = {len(R)} tests")
    print(f"  Benjamini-Hochberg FDR q={FDR_Q} across the WHOLE family -> {R['survives'].sum()} survive")
    print(f"  max |AUC - 0.50| = {(R['auc']-.5).abs().max():.3f}")
    srv = R[R["survives"]].sort_values("auc", key=lambda z: -(z - .5).abs())
    if len(srv):
        print(f"\n  {'feature':<22}{'target':<14}{'AUC':>7}{'AUCoos':>8}{'p':>11}")
        for _, r in srv.head(20).iterrows():
            print(f"  {r['feature']:<22}{r['target']:<14}{r['auc']:7.3f}{r['auc_oos']:8.3f}{r['p']:11.1e}")
    else:
        print("\n  NOTHING SURVIVES.")
    return R

if 10 in STAGES:
    R_PRE = screen(PRE_FEATS, "STAGE 10 — PRE-09:30 SCREEN")
    print("\n  Nothing before the cash open predicts anything. That is a real result:")
    print("  it tells you where to stop looking.")
else:
    R_PRE = None


# ============================================================================
# STAGE 11 — MICROSTRUCTURE SCREEN + ORTHOGONALITY
# ============================================================================

if 11 in STAGES:
    R_MIC = screen(MICRO_FEATS, "STAGE 11 — 09:30-09:45 MICROSTRUCTURE SCREEN")

    sub("11b. is it all one factor?  |correlation| with o15_ret_rel")
    cc = d[[f for f in MICRO_FEATS if f in d.columns]].corr()["o15_ret_rel"].abs().sort_values(ascending=False)
    for lab, m in (("|r| >= 0.50 (redundant)", cc[cc >= .5].drop("o15_ret_rel", errors="ignore")),
                   ("0.20-0.50 (partly indep)", cc[(cc < .5) & (cc >= .2)]),
                   ("|r| < 0.20 (orthogonal)", cc[cc < .2])):
        print(f"\n  {lab}  n={len(m)}")
        print("    " + ", ".join(f"{k} {v:.2f}" for k, v in m.items()))

    sub("11c. orthogonal signal — residualise on o15_ret_rel + o15_pos, retest on T3")
    dev = d[d["period"].isin(["IS", "VAL"])]
    core = ["o15_ret_rel", "o15_pos"]
    res = []
    for f in MICRO_FEATS:
        if f in core or f not in d.columns:
            continue
        s = dev[core + [f, "T3_brk_high"]].dropna()
        if len(s) < 500:
            continue
        A = np.c_[np.ones(len(s)), s[core].to_numpy()]
        resid = s[f].to_numpy() - A @ np.linalg.lstsq(A, s[f].to_numpy(), rcond=None)[0]
        if resid.std() < 1e-9:
            continue
        y = s["T3_brk_high"].to_numpy()
        w = logit_fit(resid.reshape(-1, 1), y)
        lr = 2 * (loglik(y, logit_pred(resid.reshape(-1, 1), w)) - loglik(y, np.full(len(y), y.mean())))
        res.append((f, auc_of(y, logit_pred(resid.reshape(-1, 1), w)), 1 - sps.chi2.cdf(max(lr, 0), 1)))
    res.sort(key=lambda z: z[2])
    ok = bh_fdr(np.array([p for _, _, p in res]))
    print(f"\n  {'feature':<22}{'residual AUC':>14}{'p':>11}   survives?")
    for (f, a, p), s_ in list(zip(res, ok))[:10]:
        print(f"  {f:<22}{a:14.3f}{p:11.2e}   {'YES' if s_ else 'no'}")
    print(f"\n  {ok.sum()} of {len(res)} carry orthogonal signal.")

    sub("11d. multivariate walk-forward on T3")
    tr, va, te = (d[d["period"] == p] for p in ("IS", "VAL", "OOS"))
    SETS = {"o15_pos alone": ["o15_pos"], "o15_ret_rel alone": ["o15_ret_rel"],
            "both": ["o15_ret_rel", "o15_pos"],
            "+ volume shape": ["o15_ret_rel", "o15_pos", "vol_up_share", "vol_front"],
            "+ timing": ["o15_ret_rel", "o15_pos", "vol_up_share", "vol_front", "t_hi_15", "t_lo_15"],
            "+ path shape": ["o15_ret_rel", "o15_pos", "vol_up_share", "vol_front", "t_hi_15",
                             "t_lo_15", "efficiency", "chop", "max_run"],
            "everything": [f for f in MICRO_FEATS if f in d.columns]}
    print(f"\n  {'feature set':<22}{'trAUC':>8}{'vaAUC':>8}{'teAUC':>8}{'teLL':>9}{'teBrier':>10}")
    for nm, fs in SETS.items():
        s = tr[fs + ["T3_brk_high"]].dropna()
        lo_, hi_ = np.percentile(s[fs], .5, axis=0), np.percentile(s[fs], 99.5, axis=0)
        w = logit_fit(np.clip(s[fs].to_numpy(), lo_, hi_), s["T3_brk_high"].to_numpy())
        o_ = []
        for ss_ in (tr, va, te):
            q = ss_[fs + ["T3_brk_high"]].dropna()
            p = logit_pred(np.clip(q[fs].to_numpy(), lo_, hi_), w)
            y = q["T3_brk_high"].to_numpy()
            o_.append((auc_of(y, p), -loglik(y, p) / len(q), np.mean((p - y) ** 2)))
        print(f"  {nm:<22}{o_[0][0]:8.3f}{o_[1][0]:8.3f}{o_[2][0]:8.3f}{o_[2][1]:9.4f}{o_[2][2]:10.4f}")
    print("\n  Judge on TEST log-loss and Brier, not train/val AUC. Anything that improves")
    print("  the first two and worsens the last two is overfitting.")
else:
    R_MIC = None


# ============================================================================
# STAGE 12 — BULL/BEAR ASYMMETRY
# ============================================================================

if 12 in STAGES:
    hdr("STAGE 12 — IS THE BEARISH SIDE A MIRROR OF THE BULLISH SIDE?")
    r = d[d["ib_brk"].isin(["high", "low"])]
    a = r[r["ib_eflow"] == 1.0]   # low formed first
    b = r[r["ib_eflow"] == 0.0]   # high formed first
    ka = (a["ib_brk"] == "high").sum(); kb = (b["ib_brk"] == "low").sum()
    pa, pb = ka / len(a), kb / len(b)
    la, ha = wilson(ka, len(a)); lb, hb = wilson(kb, len(b))
    print(f"\n  {'condition':<38}{'n':>6}{'opposite breaks 1st':>22}{'by 16:00':>11}")
    print(f"  {'IB LOW first  -> high breaks first':<38}{len(a):6d}{pa*100:14.1f}%"
          f"  [{la:.1f},{ha:.1f}]{d[d.ib_eflow==1.0]['ib_h_ses'].mean()*100:10.1f}%")
    print(f"  {'IB HIGH first -> low breaks first':<38}{len(b):6d}{pb*100:14.1f}%"
          f"  [{lb:.1f},{hb:.1f}]{d[d.ib_eflow==0.0]['ib_l_ses'].mean()*100:10.1f}%")
    z = (pa - pb) / np.sqrt(pa * (1 - pa) / len(a) + pb * (1 - pb) / len(b))
    print(f"\n  difference {(pa-pb)*100:+.1f}pp   z={z:.2f}   p={2*(1-sps.norm.cdf(abs(z))):.4f}")
    print("  The bullish version looks stronger, but that is the 54% bull base rate.")

    sub("12b. with IB close half — deviation from base is what matters")
    base = r["brk_high"].mean() * 100
    print(f"  {'ib_ef':<7}{'IB close':<10}{'n':>6}{'1st brk high%':>15}{'dev from base':>15}")
    for ef, efl in ((1.0, "low"), (0.0, "high")):
        for hf, hfl in ((True, "upper"), (False, "lower")):
            g = r[(r["ib_eflow"] == ef) & ((r["ib_pos"] >= .5) == hf)]
            if len(g) < 50:
                continue
            v = g["brk_high"].mean() * 100
            print(f"  {efl:<7}{hfl:<10}{len(g):6d}{v:15.1f}{v-base:+15.1f}")
    print("\n  The most informative cell in the study is the BEARISH one.")

    sub("12c. is the continuous signal symmetric?")
    for f, lab in (("o15_pos", "15-min close position"), ("o30_pos", "10:00 pos in ORB-30")):
        s = d.dropna(subset=[f]).copy()
        s["b"] = pd.qcut(s[f], 10, labels=False)
        print(f"\n  {lab}")
        print(f"    {'target':<22}{'D1':>7}{'D10':>7}{'base':>7}{'bear side':>11}{'bull side':>11}{'asym':>8}")
        for t, tl in TARGETS.items():
            bse = s[t].mean() * 100
            d1 = s[s["b"] == 0][t].mean() * 100
            d10 = s[s["b"] == 9][t].mean() * 100
            print(f"    {tl:<22}{d1:7.1f}{d10:7.1f}{bse:7.1f}{bse-d1:+11.1f}{d10-bse:+11.1f}{(bse-d1)-(d10-bse):+8.1f}")
    print("\n  Every row favours the bearish side: in a market that rises 54% of days, a")
    print("  bearish signal must overcome the drift to fire, so it carries more information.")


# ============================================================================
# STAGE 13 — POST-IB BREAK BACKTEST
# ============================================================================

COST_PTS = 2 * SLIP_TICKS * TICK

def run_bt(days, sm, tm, entry_buf=TICK):
    """Resting stop orders 1 tick beyond both IB extremes; first fill is the
    trade. Stop/target in IB-range units, time-stop 16:00. Stop and target in
    the same 1-min bar resolves as a LOSS (conservative).
    R is NET of costs - do not compute it gross and call it net."""
    out = []
    for s, row in days.iterrows():
        if s not in PATH_POST:
            continue
        hh, ll, cc = PATH_POST[s]
        side = row["ib_brk"]
        R = row["ib_rng"]
        if side not in ("high", "low") or not np.isfinite(R) or R <= 0:
            continue
        if side == "high":
            ent = row["ib_h"] + entry_buf
            k = np.nonzero(hh >= ent)[0]
            if not k.size:
                continue
            i0 = k[0]
            stop, targ = ent - sm * R, ent + tm * R
            ks = np.nonzero(ll[i0:] <= stop)[0]
            kt = np.nonzero(hh[i0:] >= targ)[0]
        else:
            ent = row["ib_l"] - entry_buf
            k = np.nonzero(ll <= ent)[0]
            if not k.size:
                continue
            i0 = k[0]
            stop, targ = ent + sm * R, ent - tm * R
            ks = np.nonzero(hh[i0:] >= stop)[0]
            kt = np.nonzero(ll[i0:] <= targ)[0]
        a = ks[0] if ks.size else 10 ** 9
        b = kt[0] if kt.size else 10 ** 9
        if b < a:
            pnl = tm * R
        elif a < b:
            pnl = -sm * R
        elif a == b == 10 ** 9:
            pnl = (cc[-1] - ent) if side == "high" else (ent - cc[-1])
        else:
            pnl = -sm * R
        out.append({"session": s, "side": side, "period": row["period"], "szq": row["szq"],
                    "R": (pnl - COST_PTS) / (sm * R), "pnl_pts": pnl - COST_PTS})
    return pd.DataFrame(out)


if 13 in STAGES:
    hdr("STAGE 13 — POST-IB BREAK BACKTEST")
    base_days = d[d["ib_brk"].isin(["high", "low"])].dropna(subset=["ib_rng", "szq"])
    wide = base_days[base_days["szq"] == "Q4"]

    sub("13a. stop/target surface, ALL breaks")
    print(f"  {'stop':>6}{'targ':>6}{'n':>6}{'win%':>7}{'meanR':>9}{'t':>7}{'totR':>9}")
    for sm in (0.3, 0.5, 0.75, 1.0):
        for tm in (0.3, 0.5, 0.75, 1.0, 1.5):
            rr = run_bt(base_days, sm, tm)
            t = rr["R"].mean() / (rr["R"].std(ddof=1) / np.sqrt(len(rr)))
            print(f"  {sm:6.2f}{tm:6.2f}{len(rr):6d}{(rr['R']>0).mean()*100:7.1f}"
                  f"{rr['R'].mean():+9.3f}{t:7.1f}{rr['R'].sum():+9.1f}")

    sub("13b. stop/target surface, WIDEST IB QUARTILE (ib_rng_rel top 25%, known 10:30)")
    print(f"  {'stop':>6}{'targ':>6}{'n':>6}{'win%':>7}{'meanR':>9}{'t':>7}   IS / VAL / OOS")
    for sm in (0.3, 0.5, 0.75):
        for tm in (0.5, 1.0, 1.5, 2.0):
            rr = run_bt(wide, sm, tm)
            t = rr["R"].mean() / (rr["R"].std(ddof=1) / np.sqrt(len(rr)))
            pp = "  ".join(f"{rr[rr['period']==p]['R'].mean():+.3f}" for p in ("IS", "VAL", "OOS"))
            print(f"  {sm:6.2f}{tm:6.2f}{len(rr):6d}{(rr['R']>0).mean()*100:7.1f}"
                  f"{rr['R'].mean():+9.3f}{t:7.1f}   {pp}")

    RR = run_bt(base_days, STOP_MULT, TARG_MULT)
    RW = run_bt(wide, STOP_MULT, TARG_MULT)

    sub(f"13c. walk-forward by IB size quartile (stop {STOP_MULT}, target {TARG_MULT})")
    print(f"  {'cut':<8}{'period':<8}{'n':>6}{'win%':>7}{'meanR':>9}{'t':>7}{'totR':>9}")
    for lab, s_ in [("ALL", RR)] + [(q, RR[RR["szq"] == q]) for q in ["Q1", "Q2", "Q3", "Q4"]]:
        for per in ["IS", "VAL", "OOS", "ALL"]:
            g = s_ if per == "ALL" else s_[s_["period"] == per]
            if len(g) < 40:
                continue
            t = g["R"].mean() / (g["R"].std(ddof=1) / np.sqrt(len(g)))
            print(f"  {lab:<8}{per:<8}{len(g):6d}{(g['R']>0).mean()*100:7.1f}"
                  f"{g['R'].mean():+9.3f}{t:7.1f}{g['R'].sum():+9.1f}")
        print()

    sub("13d. break direction split (widest quartile)")
    for dirn in ("high", "low"):
        g = RW[RW["side"] == dirn]
        t = g["R"].mean() / (g["R"].std(ddof=1) / np.sqrt(len(g)))
        print(f"  brk={dirn:<5} n={len(g):4d}  win {(g['R']>0).mean()*100:5.1f}%"
              f"  meanR {g['R'].mean():+.3f}  t={t:5.1f}")
    print("\n  MFE:MAE ratios favour downside breaks, but they ignore path order.")
    print("  With real stops the two sides are statistically indistinguishable. Trade both.")

    sub("13e. year by year")
    for x in (RR, RW):
        x["yr"] = pd.to_datetime(x["session"]).dt.year
    print(f"  {'year':<7}{'n_all':>7}{'R_all':>9}{'n_wide':>9}{'R_wide':>9}")
    for y in sorted(RR["yr"].unique()):
        aa, ww = RR[RR["yr"] == y], RW[RW["yr"] == y]
        print(f"  {y:<7}{len(aa):7d}{aa['R'].mean():+9.3f}{len(ww):9d}"
              + (f"{ww['R'].mean():+9.3f}" if len(ww) else " " * 9))
    print(f"\n  profitable years: all "
          f"{sum(RR[RR['yr']==y]['R'].mean()>0 for y in RR['yr'].unique())}/{RR['yr'].nunique()}"
          f"   wide {sum(RW[RW['yr']==y]['R'].mean()>0 for y in RW['yr'].unique())}/{RW['yr'].nunique()}")

    sub("13f. stationary block bootstrap")
    for x, lab in ((RR, "all breaks"), (RW, "widest quartile")):
        v = x.sort_values("session")["R"].to_numpy()
        lo, hi, p0 = block_bootstrap(v)
        print(f"  {lab:<18} n={len(v):5d}  meanR {v.mean():+.3f}"
              f"  95% CI [{lo:+.3f}, {hi:+.3f}]  P(<=0)={p0*100:.1f}%")
    print("\n  The narrowest quartile LOSES money. The width filter is not an optimisation -")
    print("  it is the difference between a setup and no setup.")
else:
    RR = RW = None


# ============================================================================
# STAGE 14 — FULL ATTRIBUTE RATE TABLE
# ============================================================================

if 14 in STAGES:
    hdr("STAGE 14 — EVERY ATTRIBUTE: BOTTOM DECILE -> TOP DECILE")
    print("  Base rates:  " + " | ".join(f"{TARGETS[t]} {d[t].mean()*100:.1f}%" for t in TARGETS))
    print("  ** = 09:30-09:45   $$ = after 09:45   else pre-09:30")
    rows = []
    for f in PRE_FEATS + MICRO_FEATS + ["o30_pos", "ib_rng_rel"]:
        if f not in d.columns:
            continue
        s = d[f].dropna()
        if len(s) < 500 or s.std() == 0:
            continue
        binary = s.nunique() <= 2
        pfx = "$$ " if f in ("o30_pos", "ib_rng_rel") else ("** " if f in MICRO_FEATS else "")
        r = {"feature": pfx + f, "n": len(s)}
        for t in TARGETS:
            q = d[[f, t]].dropna()
            if len(q) < 400 or q[f].std() == 0:
                r[f"{t}_lo"] = r[f"{t}_hi"] = r[f"{t}_spread"] = np.nan
                continue
            if binary:
                lo_ = q[q[f] <= q[f].min()][t].mean() * 100
                hi_ = q[q[f] > q[f].min()][t].mean() * 100
            else:
                try:
                    b = pd.qcut(q[f], 10, labels=False, duplicates="drop")
                except Exception:
                    r[f"{t}_lo"] = r[f"{t}_hi"] = r[f"{t}_spread"] = np.nan
                    continue
                lo_ = q[b == b.min()][t].mean() * 100
                hi_ = q[b == b.max()][t].mean() * 100
            r[f"{t}_lo"], r[f"{t}_hi"], r[f"{t}_spread"] = lo_, hi_, hi_ - lo_
        sp = [abs(r.get(f"{t}_spread", np.nan)) for t in TARGETS]
        r["max_spread"] = np.nanmax(sp) if not all(np.isnan(sp)) else np.nan
        rows.append(r)
    ATTR = pd.DataFrame(rows).sort_values("max_spread", ascending=False)
    print(f"\n{'attribute':<28}{'n':>6} | " + " | ".join(f"{TARGETS[t]:^18}" for t in TARGETS))
    for _, r in ATTR.iterrows():
        cells = []
        for t in TARGETS:
            sp = r.get(f"{t}_spread")
            cells.append(f"{'--':>18}" if pd.isna(sp)
                         else f"{r[f'{t}_lo']:5.1f}{r[f'{t}_hi']:6.1f}{sp:+7.1f}")
        print(f"{r['feature']:<28}{int(r['n']):6d} | " + " | ".join(cells))
    print(f"\n  {len(ATTR)} attributes")
else:
    ATTR = None


# ============================================================================
# STAGE 15 — EXPORTS
# ============================================================================

hdr("STAGE 15 — EXPORTS")
os.makedirs(OUT_DIR, exist_ok=True)
outs = [("nq_features.csv", d)]
if ATTR is not None: outs.append(("attribute_rates.csv", ATTR))
if R_PRE is not None: outs.append(("screen_pre0930.csv", R_PRE))
if R_MIC is not None: outs.append(("screen_micro.csv", R_MIC))
if RW is not None: outs.append(("trades_wide.csv", RW))
if RR is not None: outs.append(("trades_all.csv", RR))
if LADDER_SUM is not None: outs.append(("ladder_summary.csv", LADDER_SUM))
for nm, obj in outs:
    p = os.path.join(OUT_DIR, nm)
    obj.to_csv(p, index=(nm == "nq_features.csv"))
    print(f"  {p}   ({obj.shape[0]:,} x {obj.shape[1]})")

hdr("THE SETUP")
print("""  NQ IB break, wide-day filter
  ----------------------------
  1. At 10:30 compute the IB high/low/range and
     ib_rng_rel = IB range % / 60-day trailing median prior-day range %.
  2. Trade only if ib_rng_rel is in the top quartile. Recalculate the threshold
     on a TRAILING basis - the full-sample cut used here peeks at the future.
  3. Resting stop orders 1 tick beyond both IB extremes. First fill is the
     trade; cancel the other.
  4. Stop 0.30 x IB range. Target 1.0-2.0 x IB range.
  5. Time-stop at 16:00. Both directions.

  ~55 trades/year, ~39% win rate, roughly +0.16R per trade net of 1 tick/side.
  A 39% win rate means long losing runs - the worst historical streak was 9 and
  the worst drawdown 17R. Size for worse than that.

  Single instrument, single regime (2013-2025 is one bull market). The highest
  value next test is ES: change DATA_PATH and rerun.""")
print()


STAGE 1 — LOADING 1-MIN OHLCV
  C:\Users\tammm\NQ and ES analysis\nq_15y_1min.csv
  4,161,727 bars  |  2013-01-02 06:00:00 -> 2025-03-14 16:59:00
  3,139 session labels

STAGE 2/3 — BUILDING DAY-LEVEL AND MICROSTRUCTURE FEATURES
  Asia 18:00-02:00 | London 02:00-08:00 | 08:00 candle | 08:00-09:30 | 06:00-09:00
  3,025 days x 265 cols  |  IS=1484  VAL=995  OOS=546
  base rates: T1_ny_bull=54.4%  T2_ib_eflow=50.3%  T3_brk_high=54.2%  T4_hod_first=46.9%

STAGE 4 — CACHING 1-MIN PATHS
  10:30-16:00 raw paths: 3,033   |  3,025 of the 3,025 feature days
  have no qualifying path and are excluded from the backtest: half-sessions (Jul 3,
  Dec 24, day after Thanksgiving) close at 13:15, so a 16:00 time-stop is meaningless.

STAGE 5 — DOES THE ORB EXTREME ORDER SURVIVE INTO THE IB?

--------------------------------------------------------------------------------------------------------
  5a. carry-forward concordance
-----------------------------------------------------------------------------

IndexError: index -1 is out of bounds for axis 0 with size 0